# 3 Preprocessing Full Dataset

Encode all 2000 files with the swept parameters; write the per-encoder .npy spike arrays and labels.csv.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

In [ ]:
# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

"""
Batch Spike Encoding Pipeline for DAS Microseismic Classification
==================================================================
Processes all SEGY files (Event + Noise) through all 10 spike encoders
and saves per-encoder .npy files ready for SNN training.

All parameters from confirmed multifile sweep (400 event + 400 noise):

    Encoder    Type      sp_ev%  sp_no%  sp_avg%  SNR    Params
    ─────────────────────────────────────────────────────────────────────
    Rate       Baseline  0.409   0.514   0.461    3.09×  thr=0.85
    Phase      Baseline  0.444   0.570   0.507    2.71×  thr=0.85
    TTFS       Baseline  0.444   0.570   0.507    2.71×  thr=0.85
               [Thorpe 1996; Guo et al. Frontiers 2021]
               NOTE: converges to Phase on DAS sharp-onset data (sweep finding)
    Delta-Mod  Proposed  0.269   0.339   0.304    2.63×  thr=0.85, step=0.18
               [Inose 1962; Petro TNNLS 2020 — per-trace step scaling]
    PDE        Proposed  0.672   0.455   0.563    2.17×  kappa_phi=0.5, amp_kappa=3.0
               [Gabor 1946; Bello IEEE 2005 — MAD-adaptive dual threshold]
    ATDE       Proposed  0.416   0.282   0.349    2.22×  alpha=3.0, tau_ms=165.0
               [Jayant 1970; Haykin 2002 — EMA noise floor tau=165ms]
    MASTE      Proposed  0.366   0.256   0.311    2.52×  lags=[1,3,8], alpha=2.5
               [Mallat 1989; König 1996 — multi-lag ATDE + majority vote]
    ST-MW      Proposed  0.504   0.609   0.556    5.40×  thr_t=0.18, thr_s=0.45,
               [Petro TNNLS 2020 → 2D]         wt_ms=1.0, ws=8
    AMSTE      Proposed  0.461   0.272   0.366    3.91×  alpha=3.0, lags=[1,3,8],
               DAS-ASE (main novelty)                    min_votes=2, ws=16, thr_s=0.5
    SFBE       Proposed  2.373   0.370   1.372    0.83×  ratio_thr=10.0, sta_ms=8.0
               [FEEL-SNN NeurIPS 2024; Allen 1978]  sp_ev/sp_no=6.41× (primary metric)

Normalisation routing (critical — confirmed from single-file analysis):
    norm_env    [0,1]  → Rate, Phase, TTFS
    norm_signed [-1,1] → Delta-Mod, ATDE, MASTE, ST-MW, AMSTE
    filtered   (raw)   → PDE, SFBE (analytic signal / sub-band filter)

Output structure:
    OUTPUT_ROOT/
    ├── Rate/          file_00001.npy ...
    ├── Phase/
    ├── TTFS/
    ├── Delta-Mod/
    ├── PDE/
    ├── ATDE/
    ├── MASTE/
    ├── ST-MW/
    ├── AMSTE/
    ├── SFBE/
    ├── labels.csv     (file_id, label, class_name, filename, n_traces, n_samples, dt_ms)
    └── batch_log.txt

Each .npy: float32 spike array (n_traces, n_samples).

SFBE note: SNR metric not applicable to time-multiplexed output.
Use sparsity discrimination ratio sp_ev/sp_no=6.41× as primary SFBE metric.
"""

import segyio
import numpy as np
from scipy.signal  import butter, filtfilt, hilbert
from scipy.ndimage import maximum_filter1d, minimum_filter1d
import os
import csv
import time
import traceback

# =============================================================================
# PATHS
# =============================================================================
EVENT_DIR   = config.RAW_EVENT_DIR
NOISE_DIR   = config.RAW_NOISE_DIR
OUTPUT_ROOT = config.ENCODED_DIR

# =============================================================================
# PREPROCESSING
# =============================================================================
FORCE_DT_MS      = 0.5    # confirmed: FRISCO FORGE 2024, 2kHz DAS interrogator
FILTER_LOW_HZ    = 50
FILTER_HIGH_HZ   = 250
WINDOW_START_MS  = 0
WINDOW_END_MS    = 1000
PRE_EVENT_END_MS = 200.0  # SFBE fixed LTA baseline (pre-event window)

# =============================================================================
# ENCODER PARAMETERS — confirmed from multifile sweep (400 ev + 400 no)
# =============================================================================
RNG_SEED           = 42
TIME_WINDOW_MS     = 1.0
TRACE_SPACING_M    = 4.0

# ── Shared threshold — Rate, Phase, TTFS ─────────────────────────────────────
THRESHOLD          = 0.85          # sweep confirmed: 0.85

# ── Delta-Mod ─────────────────────────────────────────────────────────────────
DELTA_MOD_THRESHOLD = 0.85         # sweep: 0.85
DELTA_MOD_STEP_SIZE = 0.18         # sweep: 0.18  ← was 0.08, updated

# ── PDE ───────────────────────────────────────────────────────────────────────
PDE_KAPPA           = 0.5          # sweep: 0.5   ← was 1.0, updated
PDE_MIN_DELTA_PHI   = 0.10
PDE_AMP_KAPPA       = 3.0          # sweep: 3.0   ← was 2.0, updated
PDE_AMP_SMOOTH_MS   = 5.0

# ── ATDE ──────────────────────────────────────────────────────────────────────
ATDE_ALPHA          = 3.0          # sweep: 3.0   ← was 2.5, updated
ATDE_BETA           = 0.01
ATDE_NOISE_WINDOW   = 50
ATDE_EMA_TAU_MS     = 165.0        # sweep: 165.0 ← was 35.0, updated

# ── MASTE ─────────────────────────────────────────────────────────────────────
MASTE_LAGS          = [1, 3, 8]    # sweep: [1,3,8] ✓
MASTE_ALPHA         = 2.5          # sweep: 2.5   ✓
MASTE_TAU_MS        = 50.0

# ── ST-MW ─────────────────────────────────────────────────────────────────────
STMW_THR_TEMPORAL   = 0.18         # sweep: 0.18  ← was 0.35, updated
STMW_THR_SPATIAL    = 0.45         # sweep: 0.45  ← was 0.20, updated
STMW_WINDOW_T       = 1.0          # sweep: 1.0ms ← was 2.0ms, updated
STMW_WINDOW_S       = 8            # sweep: 8ch   ✓

# ── AMSTE (DAS-ASE — main novelty encoder) ────────────────────────────────────
AMSTE_ALPHA         = 3.0          # MAD multiplier — per-channel threshold
AMSTE_LAGS          = [1, 3, 8]    # temporal scales: 0.5, 1.5, 4.0 ms
AMSTE_MIN_VOTES     = 2            # majority vote (≥2/3 lags)
AMSTE_WS            = 16           # spatial window: 16ch = 64m aperture
AMSTE_THR_S         = 0.5          # spatial amplitude gate

# ── SFBE (STA/LTA fixed-baseline) ────────────────────────────────────────────
SFBE_BANDS          = [(50,100),(100,150),(150,200),(200,250)]
SFBE_RATIO_THR      = 10.0         # STA/LTA detection ratio
SFBE_STA_MS         = 8.0          # short-term averaging window

ENCODER_NAMES = [
    'Rate', 'Phase', 'TTFS',
    'Delta-Mod', 'PDE', 'ATDE', 'MASTE',
    'ST-MW', 'AMSTE', 'SFBE',
]


# =============================================================================
# PREPROCESSING FUNCTIONS
# =============================================================================

def load_segy_quiet(filepath):
    with segyio.open(filepath, mode='r', ignore_geometry=True) as f:
        data = np.array([f.trace[i] for i in range(len(f.trace))])
        try:
            dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
            if not (0.001 <= dt_ms <= 10.0):
                dt_ms = FORCE_DT_MS
        except Exception:
            dt_ms = FORCE_DT_MS
    return data, dt_ms


def bandpass_filter(data, dt_ms):
    fs   = 1000.0 / dt_ms
    nyq  = 0.5 * fs
    b, a = butter(4, [np.clip(FILTER_LOW_HZ/nyq, 1e-6, 0.99),
                      np.clip(FILTER_HIGH_HZ/nyq, 1e-6, 0.99)], btype='band')
    out  = np.zeros_like(data)
    for i in range(data.shape[0]):
        out[i] = filtfilt(b, a, data[i])
    return out


def extract_time_window(data, dt_ms):
    s = max(0, int(WINDOW_START_MS / dt_ms))
    e = min(data.shape[1], int(WINDOW_END_MS / dt_ms))
    return data[:, s:e] if s < e else data


def normalize_unit(data):
    """
    Unit normalisation [0, 1] per trace.
    Used by: Rate, Phase, TTFS (amplitude/envelope encoders).
    """
    out = np.zeros_like(data, dtype=np.float64)
    for i in range(data.shape[0]):
        t = data[i] - np.mean(data[i])
        m = np.max(np.abs(t))
        out[i] = (t / m + 1) / 2 if m > 0 else 0.5
    return out


def normalize_signed(data):
    """
    Signed normalisation [-1, +1] per trace.
    Used by: Delta-Mod, ATDE, MASTE, ST-MW, AMSTE (polarity-sensitive encoders).
    Preserves negative DAS strain (rarefaction) which is lost in [0,1] mapping.
    Without this, AMSTE neg_votes ≈ 0 and polarity novelty is non-functional.
    """
    out = np.zeros_like(data, dtype=np.float64)
    for i in range(data.shape[0]):
        t = data[i] - np.mean(data[i])
        m = np.max(np.abs(t))
        out[i] = t / m if m > 0 else t
    return out


def extract_envelope(data):
    env = np.zeros_like(data)
    for i in range(data.shape[0]):
        e = np.abs(hilbert(data[i]))
        m = np.max(e)
        env[i] = e / m if m > 0 else e
    return env


def preprocess(filepath):
    """
    Full preprocessing pipeline — identical to parameter sweep.
    Returns:
        norm_env    [0,1]  → Rate, Phase, TTFS
        norm_signed [-1,1] → Delta-Mod, ATDE, MASTE, ST-MW, AMSTE
        filtered   (raw)   → PDE, SFBE
        dt_ms
    """
    data, dt_ms  = load_segy_quiet(filepath)
    filtered     = bandpass_filter(data, dt_ms)
    filtered     = extract_time_window(filtered, dt_ms)
    envelope     = extract_envelope(filtered)
    norm_env     = normalize_unit(envelope)
    norm_signed  = normalize_signed(filtered)
    return norm_env, norm_signed, filtered, dt_ms


# =============================================================================
# ENCODER FUNCTIONS
# All encoders take their designated input type (see routing above).
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# BASELINES — use norm_env [0,1]
# ─────────────────────────────────────────────────────────────────────────────

def _rate_encoding(data, dt_ms):
    """B1: Rate — Poisson firing proportional to windowed amplitude."""
    rng = np.random.default_rng(RNG_SEED)
    spw = max(1, int(TIME_WINDOW_MS / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr = data[t]
        for i in range(0, len(tr) - spw, spw):
            amp = np.mean(np.abs(tr[i:i+spw]))
            if amp > THRESHOLD:
                out[t, i:i+spw] = (
                    rng.random(spw) < np.clip(amp, 0, 1)).astype(float)
    return out


def _phase_encoding(data, dt_ms):
    """B2: Phase — amplitude maps to spike latency within each window."""
    spw = max(1, int(TIME_WINDOW_MS / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr = data[t]
        for i in range(0, len(tr) - spw, spw):
            amp = np.max(np.abs(tr[i:i+spw]))
            if amp > THRESHOLD:
                delay = int(np.clip(1-amp, 0, 1) * (spw-1))
                idx   = i + delay
                if idx < out.shape[1]:
                    out[t, idx] = 1.0
    return out


def _ttfs_encoding(data, dt_ms):
    """
    B3: TTFS — first threshold-crossing within each window.
    Inspired by: Thorpe et al. (1996); Guo et al. Frontiers Neurosci (2021).
    Encodes stimulus ONSET LATENCY (not intensity — differs from Phase).
    DAS finding: converges to Phase on FORGE sharp-onset microseismic data.
    """
    spw = max(1, int(TIME_WINDOW_MS / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr     = data[t]
        abs_tr = np.abs(tr)
        for i in range(0, len(tr) - spw, spw):
            win = abs_tr[i:i+spw]
            if not (win > THRESHOLD).any():
                continue
            first = int(np.argmax(win > THRESHOLD))
            idx   = i + first
            if idx < out.shape[1]:
                out[t, idx] = 1.0
    return out


# ─────────────────────────────────────────────────────────────────────────────
# PROPOSED — use norm_signed [-1,+1] except PDE and SFBE
# ─────────────────────────────────────────────────────────────────────────────

def _delta_mod_encoding(data, dt_ms):
    """
    P1: Delta Modulation.
    Adapted from: Inose et al. (1962); Petro et al. TNNLS (2020).
    DAS mod: step_size=0.18×tr_range (per-trace scaling for fiber attenuation).
    Input: norm_signed [-1,+1] — detects both positive and negative transients.
    """
    min_interval = max(1, int(2.0 / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr         = data[t]
        tr_range   = np.max(tr) - np.min(tr)
        delta_step = max(DELTA_MOD_STEP_SIZE * tr_range, 1e-6)
        reference  = tr[0]
        last_spike = -min_interval
        for i in range(1, len(tr)):
            diff = tr[i] - reference
            if abs(diff) >= delta_step:
                n_steps    = int(abs(diff) / delta_step)
                reference += np.sign(diff) * n_steps * delta_step
                if diff > 0 and tr[i] > DELTA_MOD_THRESHOLD:
                    if (i - last_spike) >= min_interval:
                        out[t, i]  = 1.0
                        last_spike = i
    return out


def _pde_encoding(data, dt_ms):
    """
    P2: Phase-Delta Encoding.
    Adapted from: Gabor (1946) analytic signal; Bello et al. IEEE (2005).
    DAS mod: MAD-adaptive dual threshold — amplitude AND phase change required.
    Input: filtered (raw bandpass) — Hilbert transform computed internally.
    """
    n_traces, n_samples = data.shape
    out      = np.zeros_like(data)
    smooth_k = max(3, int(PDE_AMP_SMOOTH_MS / dt_ms))
    kernel   = np.ones(smooth_k) / smooth_k
    for t in range(n_traces):
        tr = data[t]
        N  = len(tr)
        X  = np.fft.fft(tr)
        Z  = np.zeros(N, dtype=complex)
        Z[0] = X[0]
        if N % 2 == 0:
            Z[1:N//2]  = 2.0 * X[1:N//2]
            Z[N//2]    = X[N//2]
        else:
            Z[1:(N+1)//2] = 2.0 * X[1:(N+1)//2]
        z       = np.fft.ifft(Z)
        env_raw = np.abs(z)
        env     = np.convolve(env_raw, kernel, mode='same')
        p25     = np.percentile(env, 25)
        mad_env = np.median(np.abs(env - np.median(env)))
        mu_amp  = p25 + PDE_AMP_KAPPA * 1.4826 * mad_env
        dphi    = np.angle(z[1:] * np.conj(z[:-1]))
        abs_dp  = np.abs(dphi)
        med_dp  = np.median(abs_dp)
        mad_dp  = np.median(np.abs(abs_dp - med_dp))
        thr_dp  = max(med_dp + PDE_KAPPA * 1.4826 * mad_dp, PDE_MIN_DELTA_PHI)
        for n in range(len(dphi)):
            if env[n+1] >= mu_amp and abs(dphi[n]) > thr_dp:
                out[t, n+1] = 1.0
    return out


def _atde_encoding(data, dt_ms):
    """
    P3: Adaptive Threshold Delta Encoding.
    Adapted from: Jayant (1970); Haykin 'Adaptive Filter Theory' (2002).
    DAS mod: EMA tau=165ms prevents event arrival from raising its own threshold.
    Input: norm_signed [-1,+1].
    """
    n_traces, n_samples = data.shape
    out       = np.zeros_like(data)
    W         = ATDE_NOISE_WINDOW
    alpha_ema = 1.0 - np.exp(-(dt_ms / ATDE_EMA_TAU_MS))
    for t in range(n_traces):
        tr    = data[t]
        diffs = np.diff(tr)
        init_len     = min(W, len(diffs))
        sigma_sq_ema = np.var(diffs[:init_len]) if init_len > 1 else 1e-6
        for n in range(1, n_samples):
            dx = diffs[n-1]
            if n < W:
                sigma_sq_ema = alpha_ema * dx**2 + (1-alpha_ema) * sigma_sq_ema
                continue
            sigma_sq_ema = alpha_ema * dx**2 + (1-alpha_ema) * sigma_sq_ema
            theta        = ATDE_ALPHA * max(np.sqrt(sigma_sq_ema), 1e-9) + ATDE_BETA
            if abs(dx) > theta:
                out[t, n] = 1.0
    return out


def _maste_encoding(data, dt_ms):
    """
    P4: Multi-lag Adaptive Spike Timing Encoding.
    Adapted from: Mallat (1989) wavelets; König et al. (1996) coincidence detect.
    DAS mod: ATDE per lag [0.5, 1.5, 4.0 ms] + majority vote ≥2/3.
    Input: norm_signed [-1,+1].
    """
    n_traces, n_samples = data.shape
    vote_count = np.zeros((n_traces, n_samples), dtype=np.int32)
    W          = ATDE_NOISE_WINDOW
    alpha_ema  = 1.0 - np.exp(-(dt_ms / MASTE_TAU_MS))
    for lag in MASTE_LAGS:
        out = np.zeros_like(data)
        for t in range(n_traces):
            tr    = data[t]
            diffs = tr[lag:] - tr[:-lag]
            init_len     = min(W, len(diffs))
            sigma_sq_ema = np.var(diffs[:init_len]) if init_len > 1 else 1e-6
            for n in range(lag, n_samples):
                dx = diffs[n-lag]
                if n < W + lag:
                    sigma_sq_ema = alpha_ema * dx**2 + (1-alpha_ema) * sigma_sq_ema
                    continue
                sigma_sq_ema = alpha_ema * dx**2 + (1-alpha_ema) * sigma_sq_ema
                theta        = MASTE_ALPHA * max(np.sqrt(sigma_sq_ema), 1e-9) + ATDE_BETA
                if abs(dx) > theta:
                    out[t, n] = 1.0
        vote_count += out.astype(np.int32)
    return (vote_count >= 2).astype(np.float32)


def _stmw_encoding(data, dt_ms):
    """
    P5: Spatio-Temporal Moving-Window Encoding.
    Adapted from: Moving-Window (MW), Petro et al. TNNLS (2020) → 2D for DAS.
    Joint temporal × spatial amplitude-change gate suppresses isolated noise.
    Vectorised spatial condition via scipy.ndimage (no channel loop).
    Input: norm_signed [-1,+1].
    Sweep best: SNR=5.40× — highest of all 10 encoders.
    """
    n_tr, n_smp = data.shape
    spw_t = max(1, int(STMW_WINDOW_T / dt_ms))
    spw_s = max(2, STMW_WINDOW_S)
    n_bins = n_smp // spw_t

    # Temporal condition (vectorised)
    trimmed  = data[:, :n_bins * spw_t]
    binned   = trimmed.reshape(n_tr, n_bins, spw_t)
    abs_b    = np.abs(binned)
    da_t     = abs_b.max(axis=2) - abs_b.min(axis=2)
    peak_off = np.argmax(abs_b, axis=2)
    starts   = np.arange(n_bins) * spw_t
    spike_t  = np.clip(starts[np.newaxis, :] + peak_off, 0, n_smp - 1)
    t_mids   = (starts + spw_t // 2).clip(0, n_smp - 1)

    # Spatial condition (scipy sliding filter — vectorised, no channel loop)
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=spw_s, axis=0, mode='nearest')
    spat_min = minimum_filter1d(abs_data, size=spw_s, axis=0, mode='nearest')
    da_s     = (spat_max - spat_min)[:, t_mids]

    both_met = (da_t > STMW_THR_TEMPORAL) & (da_s > STMW_THR_SPATIAL)
    out      = np.zeros((n_tr, n_smp), dtype=np.float32)
    tr_ii, b_ii = np.where(both_met)
    if len(tr_ii):
        out[tr_ii, spike_t[tr_ii, b_ii]] = 1.0
    return out


def _amste_encoding(data, dt_ms):
    """
    P6: AMSTE — Adaptive Multi-Scale Spatio-Temporal Delta Encoder (DAS-ASE).
    Novel encoder combining 4 DAS-specific properties:

    Step 1: Per-channel MAD-adaptive threshold
        θ_c = α × 1.4826 × MAD(X[c,:])
        (inspired by PhaseNet-DAS Nature Comms 2023)

    Step 2: Bidirectional polarity votes across multiple timescales
        +vote: ΔX_L[c,t] > +θ_c  (compression wave)
        −vote: ΔX_L[c,t] < −θ_c  (rarefaction wave)
        (inspired by Guo et al. Frontiers 2021; arXiv 2407.09260 2024)

    Step 3: Majority vote — spike when pos_votes OR neg_votes ≥ min_votes

    Step 4: Spatial coherence gate via scipy.ndimage (extends ST-MW)
        spike survives only if da_s[c,t] > thr_s

    Input: norm_signed [-1,+1] — REQUIRED for polarity voting to function.
    With [0,1] input neg_votes ≈ 0 and polarity novelty is non-functional.
    Sweep best: SNR=3.91×, sp_ev/sp_no=1.70×, prec=0.295
    """
    n_tr, n_smp = data.shape

    # Step 1: Per-channel MAD threshold
    med_c   = np.median(data, axis=1, keepdims=True)
    mad_c   = np.median(np.abs(data - med_c), axis=1, keepdims=True)
    theta_c = np.maximum(AMSTE_ALPHA * 1.4826 * mad_c, 1e-9)

    # Steps 2+3: Multi-scale polarity vote
    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in AMSTE_LAGS:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta_c).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta_c).astype(np.int32)

    candidate = (pos_votes >= AMSTE_MIN_VOTES) | (neg_votes >= AMSTE_MIN_VOTES)

    # Step 4: Spatial coherence gate
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=AMSTE_WS, axis=0, mode='nearest')
    spat_min = minimum_filter1d(abs_data, size=AMSTE_WS, axis=0, mode='nearest')
    da_s     = spat_max - spat_min

    return (candidate & (da_s > AMSTE_THR_S)).astype(np.float32)


def _sfbe_encoding(data, dt_ms):
    """
    P7: Sub-band Frequency Band Encoding (SFBE) with fixed-baseline STA/LTA.
    Adapted from: FEEL-SNN NeurIPS 2024 (time-multiplex retained);
                  STA/LTA onset detection: Allen (1978) BSSA.

    DAS modification:
      - Seismic-physics sub-bands (task-tuned for FORGE 2024 dataset)
      - Fixed pre-event LTA baseline (0–200ms) replaces cumulative LTA
        (cumulative LTA caused sp_no > sp_ev — SNR < 1.0 in v1)
      - STA[t]/LTA_fixed > ratio_thr fires spikes selectively on energy jumps
      - Time-multiplex retained: band position encodes frequency content

    Input: filtered (raw bandpass) — sub-band filtering computed internally.
    Primary metric: sp_ev/sp_no = 6.41× (SNR not applicable to time-multiplex).
    """
    fs_hz    = 1000.0 / dt_ms
    n_tr, n_smp = data.shape
    n_bands  = len(SFBE_BANDS)
    seg_len  = n_smp // n_bands
    out      = np.zeros((n_tr, n_smp), dtype=np.float32)
    sta_w    = max(1, int(SFBE_STA_MS / dt_ms))
    pre_end  = max(1, int(PRE_EVENT_END_MS / dt_ms))  # 200ms in samples

    for b_idx, (lo, hi) in enumerate(SFBE_BANDS):
        nyq   = 0.5 * fs_hz
        b_c, a_c = butter(4, [np.clip(lo/nyq, 1e-6, 0.99),
                               np.clip(hi/nyq, 1e-6, 0.99)], btype='band')
        seg_start = b_idx * seg_len
        seg_end   = seg_start + seg_len if b_idx < n_bands-1 else n_smp

        for t in range(n_tr):
            filt_b = filtfilt(b_c, a_c, data[t])
            env_sq = np.abs(hilbert(filt_b)) ** 2

            # Fixed LTA: mean energy in pre-event baseline window (0–200ms)
            lta_fixed = max(float(env_sq[:pre_end].mean()), 1e-12)

            # STA: causal moving average
            cs  = np.cumsum(env_sq)
            sta = np.zeros(n_smp)
            sta[:sta_w] = cs[:sta_w] / np.arange(1, sta_w + 1)
            sta[sta_w:] = (cs[sta_w:] - cs[:-sta_w]) / sta_w

            # Fire where STA/LTA_fixed > ratio_thr, map to band segment
            ratio = sta / lta_fixed
            for n in range(sta_w, n_smp):
                if ratio[n] > SFBE_RATIO_THR:
                    seg_pos = seg_start + int(n * seg_len / n_smp)
                    if seg_start <= seg_pos < seg_end:
                        out[t, seg_pos] = 1.0
    return out


# =============================================================================
# ENCODE ALL 10 — correct normalisation routing
# =============================================================================

def encode_all(norm_env, norm_signed, filtered, dt_ms):
    """
    Run all 10 encoders with correct input routing.

    Routing:
        norm_env    [0,1]  → Rate, Phase, TTFS
        norm_signed [-1,1] → Delta-Mod, ATDE, MASTE, ST-MW, AMSTE
        filtered   (raw)   → PDE, SFBE
    """
    return {
        # ── Baselines [0,1] envelope input ───────────────────────────────────
        'Rate':      _rate_encoding(norm_env,    dt_ms).astype(np.float32),
        'Phase':     _phase_encoding(norm_env,   dt_ms).astype(np.float32),
        'TTFS':      _ttfs_encoding(norm_env,    dt_ms).astype(np.float32),

        # ── Proposed signed [-1,+1] input ─────────────────────────────────────
        'Delta-Mod': _delta_mod_encoding(norm_signed, dt_ms).astype(np.float32),
        'ATDE':      _atde_encoding(norm_signed,      dt_ms).astype(np.float32),
        'MASTE':     _maste_encoding(norm_signed,     dt_ms).astype(np.float32),
        'ST-MW':     _stmw_encoding(norm_signed,      dt_ms).astype(np.float32),
        'AMSTE':     _amste_encoding(norm_signed,     dt_ms).astype(np.float32),

        # ── Proposed raw filtered input ───────────────────────────────────────
        'PDE':       _pde_encoding(filtered,          dt_ms).astype(np.float32),
        'SFBE':      _sfbe_encoding(filtered,         dt_ms).astype(np.float32),
    }


# =============================================================================
# BATCH PIPELINE
# =============================================================================

def collect_files(event_dir, noise_dir):
    files = []
    idx   = 0
    for fname in sorted(f for f in os.listdir(event_dir)
                        if f.lower().endswith(('.sgy', '.segy'))):
        idx += 1
        files.append((os.path.join(event_dir, fname), 1,
                      f'file_{idx:05d}', fname))
    n_events = idx

    for fname in sorted(f for f in os.listdir(noise_dir)
                        if f.lower().endswith(('.sgy', '.segy'))):
        idx += 1
        files.append((os.path.join(noise_dir, fname), 0,
                      f'file_{idx:05d}', fname))
    n_noise = idx - n_events
    return files, n_events, n_noise


def get_processed_ids(encoder_dirs):
    sets = []
    for enc in ENCODER_NAMES:
        d = encoder_dirs[enc]
        sets.append({os.path.splitext(f)[0]
                     for f in os.listdir(d) if f.endswith('.npy')}
                    if os.path.exists(d) else set())
    return sets[0].intersection(*sets[1:]) if sets else set()


def run_batch(resume=False):
    print("=" * 70)
    print("BATCH SPIKE ENCODING PIPELINE — ALL 10 ENCODERS")
    print(f"  Preprocessing: BP {FILTER_LOW_HZ}-{FILTER_HIGH_HZ}Hz | "
          f"Win {WINDOW_START_MS}-{WINDOW_END_MS}ms")
    print(f"  Normalisation: norm_env[0,1]→B1-B3 | "
          f"norm_signed[-1,1]→P1,P3-P6 | filtered→P2,P7")
    print(f"  Params: confirmed multifile sweep (400 ev + 400 no)")
    print(f"  Event:  {EVENT_DIR}")
    print(f"  Noise:  {NOISE_DIR}")
    print(f"  Output: {OUTPUT_ROOT}")
    print(f"  Resume: {resume}")
    print("=" * 70)

    # Create output directories
    encoder_dirs = {}
    for enc in ENCODER_NAMES:
        d = os.path.join(OUTPUT_ROOT, enc)
        os.makedirs(d, exist_ok=True)
        encoder_dirs[enc] = d

    # Collect files
    print("\nScanning directories...")
    files, n_events, n_noise = collect_files(EVENT_DIR, NOISE_DIR)
    n_total = len(files)
    print(f"  Found: {n_events} event + {n_noise} noise = {n_total} files")
    if n_total == 0:
        raise RuntimeError("No SEGY files found. Check EVENT_DIR and NOISE_DIR.")

    # Resume support
    skip_ids = get_processed_ids(encoder_dirs) if resume else set()
    if resume:
        print(f"  Resume: {len(skip_ids)} done, "
              f"{n_total - len(skip_ids)} remaining")

    labels_path = os.path.join(OUTPUT_ROOT, 'labels.csv')
    log_path    = os.path.join(OUTPUT_ROOT, 'batch_log.txt')
    csv_mode    = 'a' if resume and os.path.exists(labels_path) else 'w'
    log_mode    = 'a' if resume else 'w'

    with open(labels_path, csv_mode, newline='') as csv_file, \
         open(log_path,    log_mode)               as log_file:

        writer = csv.writer(csv_file)
        if csv_mode == 'w':
            writer.writerow(['file_id', 'label', 'class_name',
                             'original_filename', 'n_traces',
                             'n_samples', 'dt_ms'])

        log_file.write(f"\n{'='*60}\n")
        log_file.write(f"Started: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file.write(f"Files: {n_total} | Resume: {resume}\n")
        log_file.write(f"Encoders: {', '.join(ENCODER_NAMES)}\n")
        log_file.write(f"{'='*60}\n\n")

        t_start   = time.time()
        n_done    = 0
        n_skipped = 0
        n_errors  = 0
        errors    = []

        for filepath, label, file_id, orig_name in files:

            if file_id in skip_ids:
                n_skipped += 1
                continue

            class_name = 'event' if label == 1 else 'noise'

            try:
                norm_env, norm_signed, filtered, dt_ms = preprocess(filepath)
                n_tr, n_smp = norm_env.shape

                encodings = encode_all(norm_env, norm_signed, filtered, dt_ms)

                for enc in ENCODER_NAMES:
                    np.save(
                        os.path.join(encoder_dirs[enc], f'{file_id}.npy'),
                        encodings[enc])

                writer.writerow([file_id, label, class_name,
                                 orig_name, n_tr, n_smp, dt_ms])
                csv_file.flush()
                n_done += 1

                elapsed   = time.time() - t_start
                rate      = n_done / elapsed if elapsed > 0 else 0
                remaining = n_total - n_skipped - n_done - n_errors
                eta       = remaining / rate if rate > 0 else 0

                if n_done % 100 == 0 or n_done <= 5:
                    print(f"  [{n_done+n_skipped+n_errors:>6d}/{n_total}]  "
                          f"{file_id}  {class_name:5s}  "
                          f"({n_tr}×{n_smp})  "
                          f"{rate:.1f} f/s  "
                          f"ETA {eta/60:.0f} min")

            except Exception as ex:
                n_errors += 1
                msg = f"ERROR [{file_id}] {orig_name}: {ex}"
                errors.append(msg)
                log_file.write(msg + "\n")
                log_file.write(traceback.format_exc() + "\n")
                log_file.flush()
                if n_errors <= 10:
                    print(f"    {msg}")
                elif n_errors == 11:
                    print("    Further errors suppressed — see batch_log.txt")

        elapsed = time.time() - t_start
        print(f"\n{'='*70}")
        print(f"BATCH COMPLETE")
        print(f"  Processed : {n_done}")
        print(f"  Skipped   : {n_skipped} (resume)")
        print(f"  Errors    : {n_errors}")
        print(f"  Time      : {elapsed/60:.1f} min  "
              f"({n_done/elapsed:.1f} f/s)" if elapsed > 0 else "")
        print(f"  Labels    : {labels_path}")
        print(f"  Log       : {log_path}")
        print(f"{'='*70}")

        log_file.write(f"\nCompleted: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file.write(f"Done={n_done} Skip={n_skipped} Err={n_errors} "
                       f"Time={elapsed:.0f}s\n")
        if errors:
            log_file.write(
                "\nErrors:\n" + "\n".join(f"  {e}" for e in errors) + "\n")

    # Output verification
    print("\nOutput verification:")
    for enc in ENCODER_NAMES:
        count = len([f for f in os.listdir(encoder_dirs[enc])
                     if f.endswith('.npy')])
        print(f"  {enc:12s}  {count:>6d} files")

    # Sample sparsity check (first non-skipped file)
    sample = next((f for f in files if f[2] not in skip_ids), None)
    if sample and n_done > 0:
        sample_id = sample[2]
        print(f"\nSample sparsity check ({sample_id}):")
        for enc in ENCODER_NAMES:
            fp = os.path.join(encoder_dirs[enc], f'{sample_id}.npy')
            if os.path.exists(fp):
                arr = np.load(fp)
                sp  = float(np.abs(arr).mean()) * 100
                print(f"  {enc:12s}  {sp:.3f}%")


# =============================================================================
# DATASET LOADER UTILITIES
# =============================================================================

def load_encoded_dataset(output_root, encoder_name, max_files=None):
    """
    Load spike arrays + labels for one encoder.
    Returns X (N, n_traces, n_samples) float32, y (N,) int, file_ids list.
    """
    import pandas as pd
    df  = pd.read_csv(os.path.join(output_root, 'labels.csv'))
    if max_files:
        df = df.head(max_files)
    enc_dir = os.path.join(output_root, encoder_name)
    X, y, ids = [], [], []
    for _, row in df.iterrows():
        fp = os.path.join(enc_dir, f"{row['file_id']}.npy")
        if os.path.exists(fp):
            X.append(np.load(fp))
            y.append(row['label'])
            ids.append(row['file_id'])
    if not X:
        raise RuntimeError(f"No files loaded for encoder '{encoder_name}'")
    X = np.stack(X, axis=0)
    y = np.array(y, dtype=int)
    print(f"Loaded {encoder_name}: X={X.shape}  "
          f"events={np.sum(y==1)}  noise={np.sum(y==0)}")
    return X, y, ids


def load_multi_encoder_dataset(output_root, encoder_names=None, max_files=None):
    """
    Load spike arrays from multiple encoders as separate channels.
    Returns X (N, n_encoders, n_traces, n_samples) float32, y (N,) int.
    """
    import pandas as pd
    if encoder_names is None:
        encoder_names = ENCODER_NAMES
    df = pd.read_csv(os.path.join(output_root, 'labels.csv'))
    if max_files:
        df = df.head(max_files)
    all_X = []
    for enc in encoder_names:
        enc_dir = os.path.join(output_root, enc)
        arrs = []
        for _, row in df.iterrows():
            fp = os.path.join(enc_dir, f"{row['file_id']}.npy")
            arrs.append(np.load(fp) if os.path.exists(fp)
                        else np.zeros((1, 1), dtype=np.float32))
        all_X.append(np.stack(arrs, axis=0))
    X = np.stack(all_X, axis=1)
    y = df['label'].values.astype(int)
    print(f"Loaded {len(encoder_names)} encoders: X={X.shape}  "
          f"events={np.sum(y==1)}  noise={np.sum(y==0)}")
    return X, y, encoder_names


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    # Set resume=True to continue after a crash without reprocessing done files
    run_batch(resume=False)